# 03 - Analisis estadistico

Pruebas de hipotesis para confirmar -- no solo intuir -- que variables se asocian con el churn, y el diagnostico de por que los campos monetarios acumulados desde el alta (`Total_Charges`, `Total_Revenue`, ...) no son confiables para puntuar clientes `Joined`.

In [1]:
import pandas as pd
from scipy.stats import chi2_contingency, mannwhitneyu

DATA_PATH = "../data/Customer_Data.csv"
TARGET_COL = "Customer_Status"
MONTHLY_CHARGE_COL = "Monthly_Charge"
INTERNET_DEPENDENT_COLS = [
    "Internet_Type",
    "Online_Security",
    "Online_Backup",
    "Device_Protection_Plan",
    "Premium_Support",
    "Streaming_TV",
    "Streaming_Movies",
    "Streaming_Music",
    "Unlimited_Data",
]

df = pd.read_csv(DATA_PATH)
df_model = df[df[TARGET_COL].isin(["Stayed", "Churned"])].copy()
df_joined = df[df[TARGET_COL] == "Joined"].copy()

# Filtrar Monthly_Charge negativo (ver 02_eda.ipynb) en ambos conjuntos: Monthly_Charge
# entra al modelo como feature numerica cruda, asi que un cliente Joined con este valor
# sucio se puntuaria con una entrada fuera de rango si no se filtra igual que en
# entrenamiento.
df_model = df_model.loc[df_model[MONTHLY_CHARGE_COL] >= 0].copy()
df_joined = df_joined.loc[df_joined[MONTHLY_CHARGE_COL] >= 0].copy()

# Imputar nulos estructurales (ver 02_eda.ipynb): estos NaN no son datos faltantes al
# azar, existen solo cuando el cliente no tiene el servicio del que dependen.
for frame in (df_model, df_joined):
    for col in INTERNET_DEPENDENT_COLS:
        frame[col] = frame[col].fillna("No Internet Service")
    frame["Multiple_Lines"] = frame["Multiple_Lines"].fillna("No Phone Service")
    frame["Value_Deal"] = frame["Value_Deal"].fillna("No Deal")

## Variables categoricas vs churn (chi-cuadrado)

H0: la variable es independiente de `Customer_Status` (Churned/Stayed).

In [2]:
categorical_cols = [
    "Contract", "Internet_Service", "Payment_Method",
    "Value_Deal", "Married", "Paperless_Billing", "Gender",
]

chi2_results = []
for col in categorical_cols:
    table = pd.crosstab(df_model[col], df_model[TARGET_COL])
    chi2, p_value, _, _ = chi2_contingency(table)
    chi2_results.append(
        {"variable": col, "chi2": chi2, "p_value": p_value, "significativo (p<0.05)": p_value < 0.05}
    )

pd.DataFrame(chi2_results).sort_values("p_value").reset_index(drop=True)

,variable,chi2,p_value,significativo (p<0.05)
0,Contract,1521.700133,0.000000e+00,True
1,Value_Deal,644.249195,5.539687e-137,True
2,Internet_Service,293.700493,7.767292e-66,True
3,Payment_Method,290.923473,6.710741e-64,True
4,Paperless_Billing,212.325729,4.270420e-48,True
5,Gender,1.416024,2.340590e-01,False
6,Married,0.418756,5.175583e-01,False


## Variables numericas vs churn (Mann-Whitney U)

Compara la distribucion de cada variable entre clientes `Churned` y `Stayed` sin asumir normalidad.

In [3]:
numeric_cols = [
    "Age", "Number_of_Referrals", "Tenure_in_Months", "Monthly_Charge",
    "Total_Charges", "Total_Refunds", "Total_Extra_Data_Charges",
    "Total_Long_Distance_Charges", "Total_Revenue",
]

churned = df_model[df_model[TARGET_COL] == "Churned"]
stayed = df_model[df_model[TARGET_COL] == "Stayed"]

mw_results = []
for col in numeric_cols:
    stat, p_value = mannwhitneyu(churned[col], stayed[col])
    mw_results.append(
        {
            "variable": col,
            "mediana Churned": churned[col].median(),
            "mediana Stayed": stayed[col].median(),
            "p_value": p_value,
            "significativo (p<0.05)": p_value < 0.05,
        }
    )

pd.DataFrame(mw_results).sort_values("p_value").reset_index(drop=True)

,variable,mediana Churned,mediana Stayed,p_value,significativo (p<0.05)
0,Total_Revenue,913.39,2966.95,2.258519e-158,True
1,Total_Charges,706.85,1941.50,8.944837e-129,True
2,Total_Long_Distance_Charges,139.80,667.40,1.644270e-124,True
3,Monthly_Charge,79.60,66.70,4.364209e-34,True
4,Age,50.00,45.00,1.554528e-15,True
5,Total_Refunds,0.00,0.00,3.860782e-05,True
6,Total_Extra_Data_Charges,0.00,0.00,1.023921e-04,True
7,Tenure_in_Months,17.00,16.00,5.172970e-01,False
8,Number_of_Referrals,7.00,8.00,5.408257e-01,False


## Hallazgo: `Total_Charges` bajo no significa riesgo real para clientes `Joined`

Un cliente `Joined` casi no ha tenido tiempo de acumular `Total_Charges`. La pregunta es si ese mismo rango bajo, dentro del set de entrenamiento, tambien tiene un churn rate mas alto -- si es asi, un modelo entrenado con esa columna va a extrapolar el patron a *todos* los clientes nuevos, no porque tengan intencion de irse, sino porque son nuevos.

In [4]:
umbral_bajo = df_joined["Total_Charges"].quantile(0.95)
print(f"Percentil 95 de Total_Charges en clientes Joined: {umbral_bajo:.2f}")

grupo_bajo = df_model[df_model["Total_Charges"] <= umbral_bajo]
churn_rate_grupo_bajo = (grupo_bajo[TARGET_COL] == "Churned").mean()
churn_rate_general = (df_model[TARGET_COL] == "Churned").mean()

print(f"% de df_model con Total_Charges <= {umbral_bajo:.2f}: {(df_model['Total_Charges'] <= umbral_bajo).mean():.1%}")
print(f"Churn rate en ese grupo bajo: {churn_rate_grupo_bajo:.1%}")
print(f"Churn rate general:          {churn_rate_general:.1%}")

Percentil 95 de Total_Charges en clientes Joined: 233.42
% de df_model con Total_Charges <= 233.42: 13.1%
Churn rate en ese grupo bajo: 71.1%
Churn rate general:          28.9%


## Conclusion

- Variables categoricas y numericas significativas: ver las tablas de arriba (`p_value < 0.05`).
- Los campos acumulados desde el alta (`Total_Charges`, `Total_Revenue`, `Total_Refunds`, `Total_Extra_Data_Charges`, `Total_Long_Distance_Charges`) tienen una asociacion real con el churn *dentro del set de entrenamiento*, pero ese rango bajo esta dominado por bajas casi inmediatas -- no son comparables para un cliente `Joined`, que es bajo en esas columnas por construccion, no por comportamiento. `04_feature_engineering.ipynb` los excluye por esta razon.